In [3]:
#2.1
#计算 P('a' | 'b')
#转移对 ('b', 'a') 在序列中出现 1 次。即 Count('b', 'a') = 1。
#一个字符 'b' 出现 2 次。即 Count('b') = 2。
#P('a' | 'b') = (1 + 1) / (2 + 3) = 2 / 5
#计算 P('c' | 'b')
#转移对 ('b', 'c') 在序列中出现 1 次。即 Count('b', 'c') = 1。
#前一个字符 'b' 出现 2 次。即 Count('b') = 2。
#P('c' | 'b') = (1 + 1) / (2 + 3) = 2 / 5*/

In [5]:
#2.2

import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除标点
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 分词
    words = text.split()
    
    # 3. 构建词汇表
    word_counts = Counter(words)
    # 按频率排序，频率相同按字母顺序（为了确定性）
    sorted_words = sorted(word_counts.keys(), key=lambda x: (-word_counts[x], x))
    word_to_id = {word: i for i, word in enumerate(sorted_words)}
    
    # 4. 生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        feature_seq = words[i : i + n]
        label_word = words[i + n]
        features.append(feature_seq)
        labels.append(label_word)
        
    return word_to_id, (features, labels)

In [7]:
#3.2
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    h_t = np.tanh(np.dot(x_t, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h)
    cache = (x_t, h_prev, h_t, W_hx, W_hh)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    x_t, h_prev, h_t, W_hx, W_hh = cache
    
    # tanh 的导数
    dtanh = dh_next * (1 - h_t**2)
    
    # 计算梯度
    db_h = np.sum(dtanh, axis=0)
    dW_hh = np.dot(dtanh.T, h_prev)
    dW_hx = np.dot(dtanh.T, x_t)
    dh_prev = np.dot(dtanh, W_hh)
    dx_t = np.dot(dtanh, W_hx)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

In [9]:
#4.2
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BiRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=False, bidirectional=True)

    def forward(self, X):
        # X shape: (seq_len, batch, input_dim)
        output, hidden = self.rnn(X)
        # output shape: (seq_len, batch, 2 * hidden_dim)
        
        # 处理最终隐藏状态
        # hidden shape: (num_layers * 2, batch, hidden_dim)
        # 我们需要拼接最后一层的前向和后向状态
        # 前向状态在 hidden[-2], 后向状态在 hidden[-1]
        final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        # final_hidden shape: (batch, 2 * hidden_dim)
        
        return output, final_hidden

In [11]:
#5.2
import numpy as np

def cbow_loss(context_indices, target_indices, W, W_out):
    """
    context_indices: (batch_size, context_size)
    target_indices: (batch_size,)
    W: (V, d)
    W_out: (d, V)
    """
    batch_size, context_size = context_indices.shape
    
    # 1. 获取上下文词向量
    # context_embeds: (batch_size, context_size, d)
    context_embeds = W[context_indices, :]
    
    # 2. 计算平均上下文向量 (隐藏层)
    # h: (batch_size, d)
    h = np.mean(context_embeds, axis=1)
    
    # 3. 计算输出分数
    # scores: (batch_size, V)
    scores = np.dot(h, W_out)
    
    # 4. 计算 Softmax 概率
    # 为了数值稳定性，减去最大值
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # 5. 计算交叉熵损失
    # 选择目标词的概率
    correct_logprobs = -np.log(probs[np.arange(batch_size), target_indices] + 1e-8) # 加小数避免log(0)
    loss = np.sum(correct_logprobs) / batch_size
    
    return loss

In [15]:
#6.2
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V):
        # Q, K, V shape: (batch, num_heads, seq_len, d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, V)
        return output
        
    def forward(self, X):
        # X shape: (seq_len, batch, d_model)
        seq_len, batch_size, _ = X.size()
        
        # 1. 线性投影并分离头
        # 首先转置为 (batch, seq_len, d_model) 以便于线性层处理
        X = X.transpose(0, 1)
        
        Q = self.W_q(X) # (batch, seq_len, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 分离头: (batch, seq_len, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 计算注意力
        attn_output = self.scaled_dot_product_attention(Q, K, V)
        # attn_output shape: (batch, num_heads, seq_len, d_k)
        
        # 3. 拼接头
        # (batch, num_heads, seq_len, d_k) -> (batch, seq_len, num_heads, d_k)
        attn_output = attn_output.transpose(1, 2).contiguous()
        # (batch, seq_len, d_model)
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)
        
        # 4. 最终线性层
        output = self.W_o(attn_output)
        
        # 转置回 (seq_len, batch, d_model)
        output = output.transpose(0, 1)
        
        return output